# Brackish: UICE-MIRNet vs ERG-UICE-MIRNet

This notebook performs a **paired inference comparison only**: the same Brackish RGB frames are passed through a trained baseline UICE-MIRNet and a trained ERG-UICE-MIRNet. It does not apply white balance, gamma correction, CLAHE, or any other post-processing.

For a fair result, train both checkpoints with the same data split, pseudo/reference targets, crop size, epochs, optimizer schedule, and seed. Brackish does not provide clean enhancement targets, so no-reference quality metrics are proxies—not proof of better object detection. Validate the detection claim afterward with a detector trained/evaluated on identical Brackish splits and labels.

In [ ]:
from pathlib import Path
import csv
import json
import random
import sys

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'model.py').is_file():
    candidate = PROJECT_DIR / 'UICE-CV' / 'files (6)'
    if candidate.is_dir():
        PROJECT_DIR = candidate
    else:
        raise RuntimeError('Run this notebook from files (6) or the repository root.')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

from model import UICEMIRNet
from erg_uice_mirnet import ERGUICEMIRNet
from quality_metrics import METRIC_NAMES, image_metrics


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Project: {PROJECT_DIR}')
print(f'Device: {DEVICE}')

## Configure the matched experiment

Set the two checkpoint paths below. `ERG_CHECKPOINT` must be a checkpoint trained with `ERGUICEMIRNet`, not a baseline checkpoint loaded into ERG with freshly initialized guidance layers. The default image directory is the repository's extracted Brackish frames.

Use `MAX_IMAGES = None` for every image. A fixed random subset is convenient for a quick run, but it must be the same subset for both models.

In [ ]:
BRACKISH_IMAGE_DIR = PROJECT_DIR.parent / 'brackish' / 'extracted_frames'
BASELINE_CHECKPOINT = PROJECT_DIR.parent / 'checkpoints' / 'brackish_baseline' / 'uice_mirnet_best.pth'
ERG_CHECKPOINT = PROJECT_DIR.parent / 'checkpoints' / 'brackish_erg' / 'erg_uice_mirnet_best.pth'
OUTPUT_DIR = PROJECT_DIR.parent / 'results' / 'brackish_erg_comparison'

# These must match the ERG checkpoint architecture used during training.
ERG_FUSION_MODE = 'feature'  # feature | reliability_weights | feature_and_reliability_weights
ERG_RELIABILITY_CHANNELS = 16

MAX_IMAGES = 100       # Set to None to evaluate every frame.
SAMPLE_SEED = 42
SAVE_OUTPUT_IMAGES = True

if not BRACKISH_IMAGE_DIR.is_dir():
    raise FileNotFoundError(f'Brackish image directory not found: {BRACKISH_IMAGE_DIR}')
for checkpoint in (BASELINE_CHECKPOINT, ERG_CHECKPOINT):
    if not checkpoint.is_file():
        raise FileNotFoundError(
            f'Checkpoint not found: {checkpoint}\n'
            'Train the corresponding model first, then update this path.'
        )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Outputs:', OUTPUT_DIR)

## Load trained checkpoints

Strict loading intentionally fails if the checkpoint architecture does not match. This prevents an accidental comparison against a partially initialized ERG model.

In [ ]:
def read_checkpoint(path):
    checkpoint = torch.load(path, map_location=DEVICE)
    if 'model_state' not in checkpoint:
        raise KeyError(f'{path} does not contain model_state')
    return checkpoint

def architecture_from_checkpoint(checkpoint):
    args = checkpoint.get('args', {})
    return int(args.get('channels', 32)), int(args.get('num_rrb', 3))

baseline_checkpoint = read_checkpoint(BASELINE_CHECKPOINT)
erg_checkpoint = read_checkpoint(ERG_CHECKPOINT)
baseline_channels, baseline_rrbs = architecture_from_checkpoint(baseline_checkpoint)
erg_channels, erg_rrbs = architecture_from_checkpoint(erg_checkpoint)

baseline_model = UICEMIRNet(channels=baseline_channels, num_rrb=baseline_rrbs).to(DEVICE)
erg_model = ERGUICEMIRNet(
    channels=erg_channels,
    num_rrb=erg_rrbs,
    reliability_channels=ERG_RELIABILITY_CHANNELS,
    fusion_mode=ERG_FUSION_MODE,
).to(DEVICE)
baseline_model.load_state_dict(baseline_checkpoint['model_state'], strict=True)
erg_model.load_state_dict(erg_checkpoint['model_state'], strict=True)
baseline_model.eval()
erg_model.eval()

print(f'Baseline: channels={baseline_channels}, RRBs={baseline_rrbs}')
print(f'ERG: channels={erg_channels}, RRBs={erg_rrbs}, mode={ERG_FUSION_MODE}')

## Matched inference helpers

The network output is clamped only when converting it to an 8-bit RGB image for saving and scoring. No enhancement operation is performed after the neural network. Images are reflect-padded to a multiple of eight solely because the original four-stream UICE-MIRNet uses three 2× downsampling levels.

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

def list_images(folder):
    files = sorted(path for path in folder.rglob('*') if path.suffix.lower() in IMAGE_EXTENSIONS)
    if not files:
        raise RuntimeError(f'No images found in {folder}')
    if MAX_IMAGES is None or MAX_IMAGES >= len(files):
        return files
    rng = random.Random(SAMPLE_SEED)
    return sorted(rng.sample(files, MAX_IMAGES))

def pil_to_tensor(image):
    array = np.asarray(image.convert('RGB'), dtype=np.float32).copy() / 255.0
    return torch.from_numpy(array.transpose(2, 0, 1)).unsqueeze(0)

def tensor_to_pil(tensor):
    array = tensor.squeeze(0).permute(1, 2, 0).detach().cpu().numpy()
    array = np.rint(np.clip(array, 0.0, 1.0) * 255.0).astype(np.uint8)
    return Image.fromarray(array, mode='RGB')

@torch.inference_mode()
def run_model(model, image):
    tensor = pil_to_tensor(image).to(DEVICE)
    _, _, height, width = tensor.shape
    pad_h = (8 - height % 8) % 8
    pad_w = (8 - width % 8) % 8
    if pad_h or pad_w:
        tensor = F.pad(tensor, (0, pad_w, 0, pad_h), mode='reflect')
    prediction = model(tensor)
    return tensor_to_pil(prediction[:, :, :height, :width])

image_paths = list_images(BRACKISH_IMAGE_DIR)
print(f'Evaluating {len(image_paths)} matched Brackish frames')

## Run the direct comparison

Each row stores scores for the original input, baseline output, and ERG output. Output images are written to separate folders with a `.png` suffix to avoid overwriting the source frames.

In [ ]:
baseline_dir = OUTPUT_DIR / 'uice_mirnet_raw'
erg_dir = OUTPUT_DIR / 'erg_uice_mirnet_raw'
if SAVE_OUTPUT_IMAGES:
    baseline_dir.mkdir(parents=True, exist_ok=True)
    erg_dir.mkdir(parents=True, exist_ok=True)

records = []
previews = []
for index, image_path in enumerate(image_paths, start=1):
    with Image.open(image_path) as opened:
        original = opened.convert('RGB')
    baseline_output = run_model(baseline_model, original)
    erg_output = run_model(erg_model, original)

    if original.size != baseline_output.size or original.size != erg_output.size:
        raise RuntimeError(f'Resolution changed for {image_path.name}')

    named_images = {
        'Original': original,
        'UICE-MIRNet': baseline_output,
        'ERG-UICE-MIRNet': erg_output,
    }
    for stage, stage_image in named_images.items():
        records.append({'image': image_path.name, 'stage': stage, **image_metrics(stage_image)})

    if SAVE_OUTPUT_IMAGES:
        output_name = f'{image_path.stem}.png'
        baseline_output.save(baseline_dir / output_name)
        erg_output.save(erg_dir / output_name)
    if len(previews) < 4:
        previews.append((image_path.name, original, baseline_output, erg_output))
    if index == 1 or index % 25 == 0 or index == len(image_paths):
        print(f'Processed {index}/{len(image_paths)}: {image_path.name}')

results = pd.DataFrame(records)
results_path = OUTPUT_DIR / 'per_image_no_postprocessing_metrics.csv'
results.to_csv(results_path, index=False)
print(f'Saved per-image metrics: {results_path}')

## Inspect paired metric changes

A positive `ERG − baseline` value is not universally better: UIQM/UCIQE are no-reference proxies, and texture-related metrics can rise with noise or oversharpening. Use the saved images and a detector evaluation alongside this table.

In [ ]:
summary = results.groupby('stage')[list(METRIC_NAMES)].agg(['mean', 'median', 'std'])
display(summary)

baseline_scores = results[results.stage == 'UICE-MIRNet'].set_index('image')[list(METRIC_NAMES)]
erg_scores = results[results.stage == 'ERG-UICE-MIRNet'].set_index('image')[list(METRIC_NAMES)]
if not baseline_scores.index.equals(erg_scores.index):
    raise RuntimeError('Baseline and ERG image sets do not match.')
paired_delta = erg_scores - baseline_scores
comparison = pd.DataFrame({
    'baseline_mean': baseline_scores.mean(),
    'erg_mean': erg_scores.mean(),
    'mean_erg_minus_baseline': paired_delta.mean(),
    'median_erg_minus_baseline': paired_delta.median(),
    'erg_higher_fraction': (paired_delta > 0).mean(),
})
comparison_path = OUTPUT_DIR / 'metric_summary_baseline_vs_erg.csv'
comparison.to_csv(comparison_path)
display(comparison)
print(f'Saved summary: {comparison_path}')

In [ ]:
fig, axes = plt.subplots(len(previews), 3, figsize=(15, 5 * len(previews)))
axes = np.atleast_2d(axes)
for row, (name, original, baseline_output, erg_output) in zip(axes, previews):
    for axis, title, image in zip(
        row,
        ('Original', 'UICE-MIRNet (raw output)', 'ERG-UICE-MIRNet (raw output)'),
        (original, baseline_output, erg_output),
    ):
        axis.imshow(image)
        axis.set_title(f'{title}\n{name}')
        axis.axis('off')
plt.tight_layout()
plt.show()

## Record experiment metadata

Keep this JSON beside your detector results. It makes clear that the comparison contains neural-network outputs only and records the exact checkpoints, sample, and fusion setting used.

In [ ]:
metadata = {
    'image_directory': str(BRACKISH_IMAGE_DIR),
    'image_count': len(image_paths),
    'sample_seed': SAMPLE_SEED if MAX_IMAGES is not None else None,
    'max_images': MAX_IMAGES,
    'baseline_checkpoint': str(BASELINE_CHECKPOINT),
    'erg_checkpoint': str(ERG_CHECKPOINT),
    'erg_fusion_mode': ERG_FUSION_MODE,
    'erg_reliability_channels': ERG_RELIABILITY_CHANNELS,
    'postprocessing': 'None. Outputs are directly from the two neural networks; clamp is only for 8-bit image export/metrics.',
    'metric_names': list(METRIC_NAMES),
    'metrics_note': 'No-reference image-quality proxies only. Compare downstream detector metrics on a matched split for the object-boundary hypothesis.',
}
metadata_path = OUTPUT_DIR / 'experiment_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print(f'Saved metadata: {metadata_path}')
print('Next validation: run the same object detector and evaluation split on original, baseline, and ERG image folders.')